In [1]:
# autoload
%load_ext autoreload
%autoreload 2

In [2]:
basedir = "/Users/rikhoekstra/develop/republic_ner_matching"
data_dir = f"{basedir}/data"
output_dir = f"{basedir}/output"



In [3]:
import os
import pandas as pd
from pathlib import Path

data_path = Path(data_dir)
df_res = pd.read_json(data_path / "enriched_resolutions_1626_1630_complete.json")
df_res_flat = pd.read_parquet(data_path / "resolutions_flat.parquet")
#entities
df_loc = pd.read_json(data_path / "LOC-entities.json")
df_per = pd.read_json(data_path / "PER-entities.json")
df_org = pd.read_json(data_path / "ORG-entities.json") 

In [4]:
df_res.volgnr = df_res.apply(lambda row: f"{str(row['date'])[:10]}_{row['resolution_index']}", axis=1)
df_res.date = pd.PeriodIndex(df_res.date, freq='D')

In [5]:
df_res.columns

Index(['file', 'date', 'volgnr', 'resolution_index', 'text', 'institutions',
       'persons', 'places', 'ships', 'secret', 'president_ids', 'deputy_ids'],
      dtype='str')

In [6]:
all_places = df_res['places'].explode()
all_places.value_counts()


places
Holland          1121
Frankrijk         559
Engeland          528
Amsterdam         504
Hertogenbosch     454
                 ... 
Bretagne            1
Blanet              1
Oost-Vlieland       1
Blauwe Sluis        1
Maasbommel          1
Name: count, Length: 1507, dtype: int64

In [7]:
all_people = df_res['persons'].explode()
all_people.value_counts()


persons
457922    674
90194     604
168184    533
580636    531
789512    492
         ... 
379051      1
295835      1
114036      1
658106      1
436378      1
Name: count, Length: 6638, dtype: int64

In [8]:

persons = all_people[all_people.notna()]
persons = persons.to_frame()
persons.columns = ['person_id']
persons.index.name = 'resolution_id'
persons.person_id = persons.person_id.astype(int)
persons.head(10)

,person_id
resolution_id,
1,791967
3,168184
3,124646
3,125152
4,494421
4,930606
4,727188
5,920048
5,729000


In [9]:
all_institutions = df_res['institutions'].explode()
all_institutions = all_institutions[all_institutions.notna()]
institutions = all_institutions.to_frame().reset_index().rename(columns={'index': 'resolution_id', 'institutions': 'institution_id'})
institutions

,resolution_id,institution_id
0,10,14
1,12,34
2,13,13
3,13,14
4,17,78
...,...,...
7452,19115,29
7453,19120,52
7454,19120,35
7455,19122,83


In [10]:
#we have to resolve places and institutions
persons_info = pd.read_json(data_path / "persons_info.json")
person_w_name = persons.merge(persons_info, how='left', left_on='person_id', right_on='Id_persoon')
person_w_name.drop(columns=['Id_persoon'], inplace=True)
print(len(person_w_name))
person_w_name.head(10)


38153


,person_id,fullname,short_name
0,791967,"Huygen, Rutger heer van Clarenbeek","Huygen, Rutger heer van Clarenbeek"
1,168184,"Schaffer, Goossen","Schaffer, Goossen"
2,124646,"Carleton, Dudley","Carleton, Dudley"
3,125152,"Vane, Henry","Vane, Henry"
4,494421,"van Randwijck tot Bemmel, Arnold","van Randwijck tot Bemmel, Arnold"
5,930606,"de Bar, Nicolas heer van Baugy","de Bar, Nicolas heer van Baugy"
6,727188,"Pissot, Jan","Pissot, Jan"
7,920048,"Boormaecker, Govert Govertsz.","Boormaecker, Govert Govertsz."
8,729000,"Roos, Gerrit Evertsz.","Roos, Gerrit Evertsz."
9,614058,"van Nieucoop, Gerrit Willemsz.","van Nieucoop, Gerrit Willemsz."


In [11]:
# now the same for institutions
import json
import pandas as pd

with open("/Users/rikhoekstra/develop/republic_ner_matching/data/instelling_info.json", "r", encoding="utf-8") as f:
    data = json.load(f)

institutions_info = pd.DataFrame(data["instelling"])
institutions_info.ID_instelling = institutions_info.ID_instelling.astype("str", errors='ignore')
# institutions = all_institutions[all_institutions.notna()]
# institutions = institutions.to_frame()
# institutions.columns = ['institution_id']
# institutions.institution_id = institutions.institution_id.astype("Int64", errors='ignore')
# institutions.index.name = 'resolution_id'
# institutions_info
institutions_w_name = institutions.merge(institutions_info[['ID_instelling', 'naam']], how='left', left_on='institution_id', right_on='ID_instelling')
institutions_w_name.drop(columns=['ID_instelling'], inplace=True)
print(len(institutions_w_name))
institutions_w_name.head(10)

7457


,resolution_id,institution_id,naam
0,10,14,Hof van Holland en Zeeland
1,12,34,Admiraliteit te Amsterdam
2,13,13,Hoge Raad van Holland en Zeeland
3,13,14,Hof van Holland en Zeeland
4,17,78,WIC (Heren Negentien/Bewindhebbers van de WIC)
5,22,35,Admiraliteit in Zeeland
6,23,52,Gewestelijke Staten van Zeeland
7,27,33,Admiraliteit op de Maze
8,27,34,Admiraliteit te Amsterdam
9,28,51,Gewestelijke Staten van Gelderland


In [12]:
df_res

,file,date,volgnr,resolution_index,text,institutions,persons,places,ships,secret,president_ids,deputy_ids
0,NihilActum.xml,NaT,NaT_0,0,Nihil Actum,[],[],[],[],False,[],[]
1,163003ap.xml,1630-04-03,1630-04-03_0,0,Het rapport van Huijgens en andere gedeputeerd...,[],[791967],[Emden],[],False,[494421],"[494421, 216739, 208391, 296035, 791967, 27788..."
2,163003ap.xml,1630-04-03,1630-04-03_1,1,De RvS zal de provincies aanschrijven de kapit...,[],[],[],[],False,[494421],"[494421, 216739, 208391, 296035, 791967, 27788..."
3,163003ap.xml,1630-04-03,1630-04-03_2,2,Schaffer rapporteert conform de resolutie van ...,[],"[168184, 124646, 125152]",[],[],False,[494421],"[494421, 216739, 208391, 296035, 791967, 27788..."
4,163003ap.xml,1630-04-03,1630-04-03_3,3,President Rantwijck deelt mee dat ambassadeur ...,[],"[494421, 930606, 727188]",[],[],False,[494421],"[494421, 216739, 208391, 296035, 791967, 27788..."
...,...,...,...,...,...,...,...,...,...,...,...,...
19129,162607juli.xml,1626-07-07,1626-07-07_11,11,"Dirck Abbas en Joris Sforcen, generaals\n van ...",[],"[404298, 93222]",[Calais],[],False,[418011],"[387460, 360496, 938346, 962163, 392307, 58561..."
19130,162607juli.xml,1626-07-07,1626-07-07_12,12,Orateur Haga schrijft d.d. Constantinopel [Ist...,[],"[640580, 1025]",[],[],False,[418011],"[387460, 360496, 938346, 962163, 392307, 58561..."
19131,162607juli.xml,1626-07-07,1626-07-07_13,13,De gedeputeerden van Holland hebben opnieuw aa...,[],[],[Holland],[],False,[418011],"[387460, 360496, 938346, 962163, 392307, 58561..."
19132,162607juli.xml,1626-07-07,1626-07-07_14,14,De gedeputeerden van Holland hebben aangedrong...,[],"[386973, 387460, 360496, 490592, 938346, 418011]",[Holland],[],False,[418011],"[387460, 360496, 938346, 962163, 392307, 58561..."


In [13]:
dated_institutions = institutions_w_name.merge(df_res[['date','volgnr']], how='left', left_on='resolution_id', right_index=True).drop_duplicates()
dated_institutions.head(10)

,resolution_id,institution_id,naam,date,volgnr
0,10,14,Hof van Holland en Zeeland,1630-04-03,1630-04-03_9
1,12,34,Admiraliteit te Amsterdam,1630-04-03,1630-04-03_11
2,13,13,Hoge Raad van Holland en Zeeland,1630-04-03,1630-04-03_12
3,13,14,Hof van Holland en Zeeland,1630-04-03,1630-04-03_12
4,17,78,WIC (Heren Negentien/Bewindhebbers van de WIC),1630-04-27,1630-04-27_2
5,22,35,Admiraliteit in Zeeland,1630-04-27,1630-04-27_7
6,23,52,Gewestelijke Staten van Zeeland,1630-04-27,1630-04-27_8
7,27,33,Admiraliteit op de Maze,1630-04-27,1630-04-27_12
8,27,34,Admiraliteit te Amsterdam,1630-04-27,1630-04-27_12
9,28,51,Gewestelijke Staten van Gelderland,1630-04-25,1630-04-25_0


In [14]:
df_org.head(10)

,id,name,category,labels,comment,links
0,O0000002,Admiraliteit te Londen,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[]
1,O0000003,Admiraliteit van Amsterdam,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[]
2,O0000004,Admiraliteit van Friesland,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[]
3,O0000005,Admiraliteit van Rotterdam,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[]
4,O0000006,Admiraliteit van Westfriesland,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[]
5,O0000007,Admiraliteit van Zeeland,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[]
6,O0000008,Admiraliteiten in Denemarken,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[]
7,O0000009,Admiraliteiten in Frankrijk,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[]
8,O0000010,Admiraliteiten in Pruissen,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[]
9,O0000011,Admiraliteiten in Zweden,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[]


In [15]:
# ok now we match these to the entities in the df_loc, df_per, df_org dataframes. We can do this by matching the names.


inst_matched = df_org[['id', 'name']].merge(institution_w_name[['institution_id', 'naam']], how='left', left_on='name', right_on='naam')
inst_matched['institution_id'].drop_duplicates()
print(len(inst_matched))
display(inst_matched.sample(10))


NameError: name 'institution_w_name' is not defined

In [ ]:
rvs_id = inst_matched[inst_matched['naam'].str.contains("Raad van State", na=False)].institution_id.iat[0]
resolutions_with_rvs = inst_matched[inst_matched['institution_id'] == rvs_id]
resolutions_with_rvs

,id,name,institution_id,naam
527,O0000167,Raad van State,31,Raad van State
528,O0000167,Raad van State,31,Raad van State
529,O0000167,Raad van State,31,Raad van State
530,O0000167,Raad van State,31,Raad van State
531,O0000167,Raad van State,31,Raad van State
...,...,...,...,...
820,O0000167,Raad van State,31,Raad van State
821,O0000167,Raad van State,31,Raad van State
822,O0000167,Raad van State,31,Raad van State
823,O0000167,Raad van State,31,Raad van State


In [16]:
org_annotations = pd.read_json(data_path / "ORG-annotations.json")

In [17]:
import json
import pandas as pd

# load ORG annotations
with open("/Users/rikhoekstra/develop/republic_ner_matching/data/ORG-annotations.json", "r", encoding="utf-8") as f:
    org_data = json.load(f)

org_df = pd.DataFrame(
    {
        "entity_nr": [item["provenance"]["target"][-1] for item in org_data],
        "inv": [int(item["reference"]["inv"]) for item in org_data],
        "tag_text": [item["reference"]["tag_text"] for item in org_data],
        "resolution_id": [item["reference"]["resolution_id"] for item in org_data],
        "paragraph_id": [item["reference"]["paragraph_id"] for item in org_data],
        "offset": [item["reference"]["offset"] for item in org_data],
        "end": [item["reference"]["end"] for item in org_data],
    }
)

# load inventory metadata
with open("/Users/rikhoekstra/develop/republic_ner_matching/data/inventory_metadata.json", "r", encoding="utf-8") as f:
    inventory_meta = json.load(f)

meta_df = (
    pd.DataFrame(inventory_meta)[["inventory_num", "year"]]
    .explode("year")
    .dropna(subset=["year"])
    .drop_duplicates()
)

# merge year onto annotations
org_df = org_df.merge(meta_df, left_on="inv", right_on="inventory_num", how="left").drop(columns=["inventory_num"])
org_df['entity_nr'] = org_df['entity_nr'].str.extract(r'(O\d+)')
org_df.head(10)

,entity_nr,inv,tag_text,resolution_id,paragraph_id,offset,end,year
0,O0011985,3097,chambre des arydes,session-3097-num-10-resolution-1,session-3097-num-10-para-3,33,51,1577
1,O0012188,3097,Conseil de Brabant,session-3097-num-100-resolution-5,session-3097-num-100-para-8,213,231,1577
2,O0012188,3097,Conseil de Brabant,session-3097-num-117-resolution-2,session-3097-num-117-para-4,80,98,1577
3,O0012188,3097,Conseil de Brabant,session-3097-num-163-resolution-7,session-3097-num-163-para-10,184,202,1577
4,O0012188,3099,Conseil de Brabant,session-3099-num-110-resolution-6,session-3099-num-110-para-8,76,94,1577
5,O0012188,3099,Conseil de Brabt,session-3099-num-26-resolution-12,session-3099-num-26-para-14,208,224,1577
6,O0012188,3099,Conseil de Brabt,session-3099-num-30-resolution-17,session-3099-num-30-para-43,28,44,1577
7,O0012188,3099,Conseil de Brabant,session-3099-num-46-resolution-13,session-3099-num-46-para-27,251,269,1577
8,O0012188,3099,Conseil de Brabant,session-3099-num-48-resolution-9,session-3099-num-48-para-27,0,18,1577
9,O0012188,3099,Conseil de Brabant,session-3099-num-59-resolution-6,session-3099-num-59-para-11,162,180,1577


In [18]:
org_annotations_window = df_org[['id', 'name']].merge(org_df, how='left', left_on='id', right_on='entity_nr')
org_annotations_window = org_annotations_window.merge(df_res_flat[['id', 'date']], how='left', left_on='resolution_id', right_on='id').drop_duplicates()
org_annotations_window.head(10)

,id_x,name,entity_nr,inv,tag_text,resolution_id,paragraph_id,offset,end,year,id_y,date
0,O0000002,Admiraliteit te Londen,O0000002,3120,Raeden vande gedepde. naer Engelant Admiralite...,session-3120-num-178-resolution-6,session-3120-num-178-para-14,29,88,1588,session-3120-num-178-resolution-6,1588-07-04
1,O0000002,Admiraliteit te Londen,O0000002,3122,Admiraliteyt van Engelant,session-3122-num-163-resolution-4,session-3122-num-163-para-9,323,348,1589,session-3122-num-163-resolution-4,1589-07-18
2,O0000002,Admiraliteit te Londen,O0000002,3126,Admiraliteyt van Engelant,session-3126-num-155-resolution-3,session-3126-num-155-para-5,1111,1136,1591,session-3126-num-155-resolution-3,1591-06-21
3,O0000002,Admiraliteit te Londen,O0000002,3190,Admiraliteyt van Engelant,session-3190-num-293-resolution-1,session-3190-num-293-para-8,109,134,1631,session-3190-num-293-resolution-1,1631-12-03
4,O0000002,Admiraliteit te Londen,O0000002,3337,Admiraliteijt van Engelandt,session-3337-num-99-resolution-8,session-3337-num-99-para-9,496,523,1698,session-3337-num-99-resolution-8,1698-05-13
5,O0000002,Admiraliteit te Londen,O0000002,3765,Admiraliteyt van Engelandt,session-3765-num-331-resolution-19,session-3765-num-331-para-38,342,368,1710,session-3765-num-331-resolution-19,1710-12-05
6,O0000002,Admiraliteit te Londen,O0000002,3810,Admiraliteyt van Engeland,session-3810-num-309-resolution-11,session-3810-num-309-para-13,329,354,1755,session-3810-num-309-resolution-11,1755-11-10
7,O0000002,Admiraliteit te Londen,O0000002,3128,grooten Admirael van Engelant,session-3128-num-160-resolution-7,session-3128-num-160-para-10,728,757,1592,session-3128-num-160-resolution-7,1592-09-10
8,O0000002,Admiraliteit te Londen,O0000002,3150,Admirael van Engelandt,session-3150-num-254-resolution-18,session-3150-num-254-para-32,592,614,1603,session-3150-num-254-resolution-18,1603-11-14
9,O0000002,Admiraliteit te Londen,O0000002,3180,Admirael van Engelandt,session-3180-num-9-resolution-5,session-3180-num-9-para-15,569,591,1621,session-3180-num-9-resolution-5,1621-01-21


In [19]:
org_date_window = org_annotations_window[org_annotations_window['year'].between(1626, 1630)]

In [20]:
org_date_window.date = pd.PeriodIndex(org_date_window.date, freq='D')

In [21]:
dated_institutions.date = pd.PeriodIndex(dated_institutions.date, freq='D')

In [22]:
org_overlap = dated_institutions.merge(org_date_window[['date','name','paragraph_id']], left_on=['date','naam'], right_on=['date','name'], how='inner')

In [23]:
print(len(org_overlap))
org_overlap.head(10)

1651


,resolution_id,institution_id,naam,date,volgnr,name,paragraph_id
0,80,31,Raad van State,1630-04-23,1630-04-23_7,Raad van State,session-3189-num-103-para-1
1,80,31,Raad van State,1630-04-23,1630-04-23_7,Raad van State,session-3189-num-103-para-2
2,81,31,Raad van State,1630-04-23,1630-04-23_8,Raad van State,session-3189-num-103-para-1
3,81,31,Raad van State,1630-04-23,1630-04-23_8,Raad van State,session-3189-num-103-para-2
4,137,31,Raad van State,1630-04-26,1630-04-26_1,Raad van State,session-3189-num-106-para-2
5,137,31,Raad van State,1630-04-26,1630-04-26_1,Raad van State,session-3189-num-106-para-2
6,137,31,Raad van State,1630-04-26,1630-04-26_1,Raad van State,session-3189-num-106-para-6
7,139,79,Generaliteitsrekenkamer,1630-04-26,1630-04-26_3,Generaliteitsrekenkamer,session-3189-num-106-para-4
8,161,31,Raad van State,1630-04-06,1630-04-06_4,Raad van State,session-3189-num-87-para-2
9,161,31,Raad van State,1630-04-06,1630-04-06_4,Raad van State,session-3189-num-87-para-8


In [ ]:
# org_overlap.to_excel(data_path / "org_overlap_1626_1630.xlsx", index=False)

In [24]:
df_res_flat.head(5)

,id,type,date,year,weekday,paragraph_texts,resolutions_text
0,session-3788-num-1-resolution-1,resolution,1733-01-02,1733,vrijdag,"[""ONtfangen een Missive van den Resident Spina...","ONtfangen een Missive van den Resident Spina, ..."
1,session-3788-num-1-resolution-10,resolution,1733-01-02,1733,vrijdag,"[""IS ter Vergaderinge gelesen de Requeste van ...",IS ter Vergaderinge gelesen de Requeste van de...
2,session-3788-num-1-resolution-11,resolution,1733-01-02,1733,vrijdag,"[""17 Ynde ter Vergaderinge getoont en geexhibe...",17 Ynde ter Vergaderinge getoont en geexhibeer...
3,session-3788-num-1-resolution-12,resolution,1733-01-02,1733,vrijdag,"[""17 Ynde ter Vergaderinge getoont ende geëxhi...",17 Ynde ter Vergaderinge getoont ende geëxhibe...
4,session-3788-num-1-resolution-13,resolution,1733-01-02,1733,vrijdag,"[""OP de Requeste van de gesamentlijcke Straatm...",OP de Requeste van de gesamentlijcke Straatmaa...


In [25]:
df_res_flat[(df_res_flat['id'].isin(rvs_resolutions_raw['resolution_id'])) & (df_res_flat['year'].between(1626,1630))]

NameError: name 'rvs_resolutions_raw' is not defined

In [26]:
org_df

,entity_nr,inv,tag_text,resolution_id,paragraph_id,offset,end,year
0,O0011985,3097,chambre des arydes,session-3097-num-10-resolution-1,session-3097-num-10-para-3,33,51,1577
1,O0012188,3097,Conseil de Brabant,session-3097-num-100-resolution-5,session-3097-num-100-para-8,213,231,1577
2,O0012188,3097,Conseil de Brabant,session-3097-num-117-resolution-2,session-3097-num-117-para-4,80,98,1577
3,O0012188,3097,Conseil de Brabant,session-3097-num-163-resolution-7,session-3097-num-163-para-10,184,202,1577
4,O0012188,3099,Conseil de Brabant,session-3099-num-110-resolution-6,session-3099-num-110-para-8,76,94,1577
...,...,...,...,...,...,...,...,...
507263,O0012275,4860,adelijcke assessor van't hoffgericht van oostv...,session-4860-num-411-resolution-3,session-4860-num-411-para-5,576,631,NaN
507264,O0012260,4860,Collegij van Stenden van oostvrieslandt,session-4860-num-42-resolution-1,session-4860-num-42-para-6,42,81,NaN
507265,O0012260,4860,heeren Gedepe: der Stenden van oostvrieslandt,session-4860-num-426-resolution-1,session-4860-num-426-para-1,235,280,NaN
507266,O0012255,4860,Gedepde Staten vande Provincie van Stadt ende ...,session-4860-num-5-resolution-2,session-4860-num-5-para-4,425,477,NaN


In [27]:
# Step 1: add flat resolution dates to every org annotation
flat_dates = df_res_flat[['id', 'date']].rename(columns={'id': 'resolution_id', 'date': 'flat_date'})
org_df_dated = org_df.merge(flat_dates, on='resolution_id', how='left')
org_df_dated['flat_date'] = pd.PeriodIndex(org_df_dated['flat_date'].astype(str), freq='D')

print(f"org_df_dated: {len(org_df_dated)} rows, {org_df_dated['flat_date'].isna().sum()} without date")
org_df_dated.head(5)


org_df_dated: 507268 rows, 0 without date


,entity_nr,inv,tag_text,resolution_id,paragraph_id,offset,end,year,flat_date
0,O0011985,3097,chambre des arydes,session-3097-num-10-resolution-1,session-3097-num-10-para-3,33,51,1577,1577-05-29
1,O0012188,3097,Conseil de Brabant,session-3097-num-100-resolution-5,session-3097-num-100-para-8,213,231,1577,1577-08-26
2,O0012188,3097,Conseil de Brabant,session-3097-num-117-resolution-2,session-3097-num-117-para-4,80,98,1577,1577-09-12
3,O0012188,3097,Conseil de Brabant,session-3097-num-163-resolution-7,session-3097-num-163-para-10,184,202,1577,1577-10-30
4,O0012188,3099,Conseil de Brabant,session-3099-num-110-resolution-6,session-3099-num-110-para-8,76,94,1577,1578-03-23


In [28]:
flat_dates

,resolution_id,flat_date
0,session-3788-num-1-resolution-1,1733-01-02
1,session-3788-num-1-resolution-10,1733-01-02
2,session-3788-num-1-resolution-11,1733-01-02
3,session-3788-num-1-resolution-12,1733-01-02
4,session-3788-num-1-resolution-13,1733-01-02
...,...,...
692151,session-3262-num-99-resolution-5,1656-05-09
692152,session-3262-num-99-resolution-6,1656-05-09
692153,session-3262-num-99-resolution-7,1656-05-09
692154,session-3262-num-99-resolution-8,1656-05-09


In [29]:
places = all_places.to_frame()
places.index.name = 'resolution_id'
places.columns = ['place_name']
places = places.loc[places.place_name.notna()]
places.head(10)

,place_name
resolution_id,
1,Emden
8,Tiel
8,Venlo
9,Wezel
11,Venetië
11,Holland
11,Amsterdam
13,Amboina
13,Engeland


In [30]:
dated_places = places.merge(df_res[['date', 'volgnr']], left_index=True, right_index=True, how='left')
dated_places.head(10)

,place_name,date,volgnr
resolution_id,,,
1,Emden,1630-04-03,1630-04-03_0
8,Tiel,1630-04-03,1630-04-03_7
8,Venlo,1630-04-03,1630-04-03_7
9,Wezel,1630-04-03,1630-04-03_8
11,Venetië,1630-04-03,1630-04-03_10
11,Holland,1630-04-03,1630-04-03_10
11,Amsterdam,1630-04-03,1630-04-03_10
13,Amboina,1630-04-03,1630-04-03_12
13,Engeland,1630-04-03,1630-04-03_12


In [31]:
loc_annotations = pd.read_json(data_path / "LOC-annotations.json")
locs = loc_annotations

In [32]:
# change org to loc

# load LOC annotations
with open("/Users/rikhoekstra/develop/republic_ner_matching/data/LOC-annotations.json", "r", encoding="utf-8") as f:
    loc_data = json.load(f)

loc_df = pd.DataFrame(
    {
        "entity_nr": [item["provenance"]["target"][-1] for item in loc_data],
        "inv": [int(item["reference"]["inv"]) for item in loc_data],
        "tag_text": [item["reference"]["tag_text"] for item in loc_data],
        "resolution_id": [item["reference"]["resolution_id"] for item in loc_data],
        "paragraph_id": [item["reference"]["paragraph_id"] for item in loc_data],
        "offset": [item["reference"]["offset"] for item in loc_data],
        "end": [item["reference"]["end"] for item in loc_data],
    }
)

# # load inventory metadata
# with open("/Users/rikhoekstra/develop/republic_ner_matching/data/inventory_metadata.json", "r", encoding="utf-8") as f:
#     inventory_meta = json.load(f)

# meta_df = (
#     pd.DataFrame(inventory_meta)[["inventory_num", "year"]]
#     .explode("year")
#     .dropna(subset=["year"])
#     .drop_duplicates()
# )

# merge year onto annotations
loc_df = loc_df.merge(meta_df, left_on="inv", right_on="inventory_num", how="left").drop(columns=["inventory_num"])
loc_df['entity_nr'] = loc_df['entity_nr'].str.extract(r'(L\d+)')

In [33]:
loc_df

,entity_nr,inv,tag_text,resolution_id,paragraph_id,offset,end,year
0,L0002328,3097,Maestricht,session-3097-num-105-resolution-8,session-3097-num-105-para-8,401,411,1577
1,L0001567,3097,Bruessele,session-3097-num-106-resolution-1,session-3097-num-106-para-3,191,200,1577
2,L0003089,3097,Stadt van Utrecht,session-3097-num-109-resolution-1,session-3097-num-109-para-1,85,102,1577
3,L0002509,3097,Ansloo in Noorwegen,session-3097-num-110-resolution-1,session-3097-num-110-para-1,46,65,1577
4,L0001768,3097,denemarken,session-3097-num-110-resolution-1,session-3097-num-110-para-1,83,93,1577
...,...,...,...,...,...,...,...,...
1383585,L0007465,4860,Oostvrieslandt,session-4860-num-94-resolution-1,session-4860-num-94-para-1,512,526,NaN
1383586,L0007465,4860,oostvrieslandt,session-4860-num-94-resolution-1,session-4860-num-94-para-1,1056,1070,NaN
1383587,L0008059,4860,inden Hage,session-4860-num-95-resolution-1,session-4860-num-95-para-2,93,103,NaN
1383588,L0008059,4860,inden Hage,session-4860-num-97-resolution-3,session-4860-num-97-para-3,79,89,NaN


In [34]:
flat_dates

,resolution_id,flat_date
0,session-3788-num-1-resolution-1,1733-01-02
1,session-3788-num-1-resolution-10,1733-01-02
2,session-3788-num-1-resolution-11,1733-01-02
3,session-3788-num-1-resolution-12,1733-01-02
4,session-3788-num-1-resolution-13,1733-01-02
...,...,...
692151,session-3262-num-99-resolution-5,1656-05-09
692152,session-3262-num-99-resolution-6,1656-05-09
692153,session-3262-num-99-resolution-7,1656-05-09
692154,session-3262-num-99-resolution-8,1656-05-09


In [35]:
loc_annotations_dated = loc_df.merge(flat_dates, on='resolution_id', how='left')
loc_annotations_dated['date'] = pd.PeriodIndex(loc_annotations_dated['flat_date'].astype(str), freq='D')
loc_annotations_dated_window = loc_annotations_dated[(loc_annotations_dated['date'] >= pd.Period('1626-01-01', freq='D')) & (loc_annotations_dated['date'] <= pd.Period('1630-12-31', freq='D'))]
loc_annotations_dated_window.head(5)

,entity_nr,inv,tag_text,resolution_id,paragraph_id,offset,end,year,flat_date,date
99338,L0001860,3185,Engelant,session-3185-num-1-resolution-1,session-3185-num-1-para-3,205,213,1626,1626-01-01,1626-01-01
99339,L0007323,3185,Vranckryck,session-3185-num-1-resolution-1,session-3185-num-1-para-4,47,57,1626,1626-01-01,1626-01-01
99340,L0007323,3185,Vranckryck,session-3185-num-1-resolution-1,session-3185-num-1-para-4,95,105,1626,1626-01-01,1626-01-01
99341,L0001365,3185,Amsterdam,session-3185-num-1-resolution-1,session-3185-num-1-para-4,249,258,1626,1626-01-01,1626-01-01
99342,L0007323,3185,Vranckryck,session-3185-num-1-resolution-1,session-3185-num-1-para-4,323,333,1626,1626-01-01,1626-01-01


In [36]:
df_loc

,id,name,geo_data,links,labels,comment
0,L0006150,Niel,"{'region': 'Europa', 'modern_country': 'België...","[{'type': 'geonames_id', 'target': '2790221'}]",NaN,NaN
1,L0002729,Riel,"{'region': 'Europa', 'modern_country': 'Nederl...","[{'type': 'geonames_id', 'target': '2748164'}]",NaN,NaN
2,L0006152,Franeker,"{'region': 'Europa', 'modern_country': 'Nederl...","[{'type': 'geonames_id', 'target': '2755845'}]",NaN,NaN
3,L0001428,Barcelona,"{'region': 'Europa', 'modern_country': 'Spanje...","[{'type': 'geonames_id', 'target': '3128760'}]",NaN,NaN
4,L0006153,Hontenisse,"{'region': 'Europa', 'modern_country': 'Nederl...","[{'type': 'geonames_id', 'target': '9883612'}]",NaN,NaN
...,...,...,...,...,...,...
2454,L0007694,Hattingen,"{'region': 'Europa', 'modern_country': 'Duitsl...","[{'type': 'geonames_id', 'target': '2909230'}]",NaN,NaN
2455,L0007695,Boekel,"{'region': 'Europa', 'modern_country': 'Nederl...","[{'type': 'geonames_id', 'target': '2758757'}]",NaN,NaN
2456,L0008393,Fernando Po,"{'region': 'Afrika', 'modern_country': 'Equato...","[{'type': 'geonames_id', 'target': '2566978'}]",NaN,Ook als: Bioko
2457,L0007696,Bar-le-Duc,"{'region': 'Europa', 'modern_country': 'Frankr...","[{'type': 'geonames_id', 'target': '3034911'}]",NaN,NaN


In [37]:
# now include normalized names for places
loc_annotations_dated_window = loc_annotations_dated_window.merge(df_loc[['id', 'name']], left_on='entity_nr', right_on='id', how='left')
loc_annotations_dated_window.head(5)

,entity_nr,inv,tag_text,resolution_id,paragraph_id,offset,end,year,flat_date,date,id,name
0,L0001860,3185,Engelant,session-3185-num-1-resolution-1,session-3185-num-1-para-3,205,213,1626,1626-01-01,1626-01-01,L0001860,Engeland
1,L0007323,3185,Vranckryck,session-3185-num-1-resolution-1,session-3185-num-1-para-4,47,57,1626,1626-01-01,1626-01-01,L0007323,Frankrijk
2,L0007323,3185,Vranckryck,session-3185-num-1-resolution-1,session-3185-num-1-para-4,95,105,1626,1626-01-01,1626-01-01,L0007323,Frankrijk
3,L0001365,3185,Amsterdam,session-3185-num-1-resolution-1,session-3185-num-1-para-4,249,258,1626,1626-01-01,1626-01-01,L0001365,Amsterdam
4,L0007323,3185,Vranckryck,session-3185-num-1-resolution-1,session-3185-num-1-para-4,323,333,1626,1626-01-01,1626-01-01,L0007323,Frankrijk


In [38]:
loc_annotations_dated_window.tag_text.value_counts()

tag_text
Amsterdam                1029
Rotterdam                 844
Vranckryck                748
Hollant                   747
Engelant                  673
                         ... 
Oostvriesslandt             1
Nederlandt                  1
artois ende Henegouwe       1
inden Haeghs                1
Zoom & Steenbergen          1
Name: count, Length: 2903, dtype: int64

In [39]:
print(f"dated_places.date.dtype: {dated_places.date.dtype}")
print(f"loc_annotations_dated_window.date.dtype: {loc_annotations_dated_window.date.dtype}")

dated_places.date.dtype: period[D]
loc_annotations_dated_window.date.dtype: period[D]


In [46]:
dated_places['date'] = pd.PeriodIndex(dated_places['date'].astype(str), freq='D')
overlap_common = set(dated_places['date']) & set(org_overlap['date'])
pd.DataFrame(list(overlap_common), columns=['date']).to_excel(data_path / "overlap_common_dates.xlsx", index=False)

In [ ]:
dated_places

,place_name,date,volgnr
resolution_id,,,
1,Emden,1630-04-03,1630-04-03_0
8,Tiel,1630-04-03,1630-04-03_7
8,Venlo,1630-04-03,1630-04-03_7
9,Wezel,1630-04-03,1630-04-03_8
11,Venetië,1630-04-03,1630-04-03_10
...,...,...,...
19128,Vlaanderen,1626-07-07,1626-07-07_10
19128,Brabant,1626-07-07,1626-07-07_10
19129,Calais,1626-07-07,1626-07-07_11


In [ ]:
place_overlap = dated_places.merge(loc_annotations_dated_window[['date','name','paragraph_id']], left_on=['date','place_name'], right_on=['date','name'], how='inner')
place_overlap


,place_name,date,volgnr,name,paragraph_id
0,Emden,1630-04-03,1630-04-03_0,Emden,session-3189-num-84-para-3
1,Venetië,1630-04-03,1630-04-03_10,Venetië,session-3189-num-84-para-11
2,Holland,1630-04-03,1630-04-03_10,Holland,session-3189-num-84-para-11
3,Amsterdam,1630-04-03,1630-04-03_10,Amsterdam,session-3189-num-84-para-11
4,Amboina,1630-04-03,1630-04-03_12,Amboina,session-3189-num-84-para-13
...,...,...,...,...,...
11843,Holland,1626-07-07,1626-07-07_1,Holland,session-3185-num-113-para-16
11844,Vlaanderen,1626-07-07,1626-07-07_10,Vlaanderen,session-3185-num-113-para-10
11845,Calais,1626-07-07,1626-07-07_11,Calais,session-3185-num-113-para-11
11846,Holland,1626-07-07,1626-07-07_13,Holland,session-3185-num-113-para-16


In [ ]:
place_overlap.to_excel(data_path / "place_overlap_1626_1630.xlsx", index=False)


In [65]:
org_df_dated

,entity_nr,inv,tag_text,resolution_id,paragraph_id,offset,end,year,flat_date
0,O0011985,3097,chambre des arydes,session-3097-num-10-resolution-1,session-3097-num-10-para-3,33,51,1577,1577-05-29
1,O0012188,3097,Conseil de Brabant,session-3097-num-100-resolution-5,session-3097-num-100-para-8,213,231,1577,1577-08-26
2,O0012188,3097,Conseil de Brabant,session-3097-num-117-resolution-2,session-3097-num-117-para-4,80,98,1577,1577-09-12
3,O0012188,3097,Conseil de Brabant,session-3097-num-163-resolution-7,session-3097-num-163-para-10,184,202,1577,1577-10-30
4,O0012188,3099,Conseil de Brabant,session-3099-num-110-resolution-6,session-3099-num-110-para-8,76,94,1577,1578-03-23
...,...,...,...,...,...,...,...,...,...
507263,O0012275,4860,adelijcke assessor van't hoffgericht van oostv...,session-4860-num-411-resolution-3,session-4860-num-411-para-5,576,631,NaN,1670-01-06
507264,O0012260,4860,Collegij van Stenden van oostvrieslandt,session-4860-num-42-resolution-1,session-4860-num-42-para-6,42,81,NaN,1664-04-23
507265,O0012260,4860,heeren Gedepe: der Stenden van oostvrieslandt,session-4860-num-426-resolution-1,session-4860-num-426-para-1,235,280,NaN,1670-09-26
507266,O0012255,4860,Gedepde Staten vande Provincie van Stadt ende ...,session-4860-num-5-resolution-2,session-4860-num-5-para-4,425,477,NaN,1664-01-08


In [ ]:
# which overlap dates coincide between places and orgs? 

overlap_dates = set(place_overlap_dated['flat_date']) & set(org_df_dated['flat_date'])
overlap_dates.

{Period('1628-07-05', 'D'),
 Period('1630-07-30', 'D'),
 Period('1630-07-19', 'D'),
 Period('1630-11-30', 'D'),
 Period('1627-10-28', 'D'),
 Period('1629-11-10', 'D'),
 Period('1629-11-21', 'D'),
 Period('1630-12-11', 'D'),
 Period('1627-02-19', 'D'),
 Period('1626-01-19', 'D'),
 Period('1626-01-08', 'D'),
 Period('1627-07-14', 'D'),
 Period('1626-06-02', 'D'),
 Period('1629-07-27', 'D'),
 Period('1629-02-21', 'D'),
 Period('1626-10-25', 'D'),
 Period('1628-11-18', 'D'),
 Period('1626-02-16', 'D'),
 Period('1630-01-17', 'D'),
 Period('1628-05-28', 'D'),
 Period('1627-04-28', 'D'),
 Period('1630-06-11', 'D'),
 Period('1630-10-23', 'D'),
 Period('1627-09-20', 'D'),
 Period('1626-08-20', 'D'),
 Period('1629-09-22', 'D'),
 Period('1629-10-03', 'D'),
 Period('1627-01-12', 'D'),
 Period('1629-01-25', 'D'),
 Period('1629-02-05', 'D'),
 Period('1629-06-08', 'D'),
 Period('1626-04-25', 'D'),
 Period('1626-05-06', 'D'),
 Period('1630-02-14', 'D'),
 Period('1628-09-30', 'D'),
 Period('1630-02-25'

In [120]:
# Step 2: build date-based resolution refs and anchor candidates for 1626-1630
anchor_window_start = 1626
anchor_window_end = 1630

flat_window = (
    df_res_flat.loc[
        df_res_flat['year'].between(anchor_window_start, anchor_window_end),
        ['id', 'date', 'year'],
    ]
    .copy()
    .sort_values(['date', 'id'])
)
flat_window['date_period'] = pd.PeriodIndex(flat_window['date'].astype(str), freq='D')
flat_window['sequence_nr'] = flat_window.groupby('date_period').cumcount() + 1
flat_window['resolution_ref'] = (
    flat_window['date_period'].astype(str)
    + '-'
    + flat_window['sequence_nr'].astype(str).str.zfill(2)
)

org_window = org_df_dated.merge(
    flat_window[['id', 'resolution_ref']],
    left_on='resolution_id',
    right_on='id',
    how='inner',
)
org_window = org_window[org_window['year'].between(anchor_window_start, anchor_window_end)].copy()
org_window['anchor_ref'] = org_window['resolution_ref']
org_window['anchor_label'] = org_window['tag_text']

print(f"Anchor candidates in {anchor_window_start}-{anchor_window_end}: {len(org_window)}")
print(f"Unique resolution refs: {org_window['anchor_ref'].nunique()}")
print(f"Unique orgs: {org_window['entity_nr'].nunique()}")
org_window[['resolution_id', 'anchor_ref', 'entity_nr', 'anchor_label', 'year', 'flat_date', 'offset', 'end']].head(10)


Anchor candidates in 1626-1630: 9678
Unique resolution refs: 6294
Unique orgs: 76


,resolution_id,anchor_ref,entity_nr,anchor_label,year,flat_date,offset,end
0,session-3186-num-115-resolution-4,1627-07-04-34,O0012259,Estats Generaulx,1627,1627-07-04,14,30
1,session-3187-num-45-resolution-1,1628-02-17-01,O0012259,Estats Generaulx,1628,1628-02-17,152,168
3,session-3185-num-28-resolution-23,1626-02-12-16,O0012259,Messieurs les Estats Generaulx,1626,1626-02-12,266,296
4,session-3186-num-110-resolution-6,1627-06-22-32,O0012259,Messieurs les Estats Generaulxle,1627,1627-06-22,71,103
5,session-3189-num-282-resolution-18,1630-11-13-10,O0012259,Messieurs les Estatz generaulx,1630,1630-11-13,393,423
6,session-3186-num-145-resolution-7,1627-08-26-29,O0012247,Staten van Brabant,1627,1627-08-26,277,295
7,session-3186-num-21-resolution-7,1627-02-04-15,O0012247,Staten van Brabant,1627,1627-02-04,398,416
8,session-3186-num-34-resolution-2,1627-02-22-02,O0012247,Staten van Brabant,1627,1627-02-22,273,291
9,session-3187-num-147-resolution-18,1628-06-14-10,O0012247,Staten van Brabant,1628,1628-06-14,341,359
10,session-3187-num-164-resolution-8,1628-07-05-23,O0012247,Staten van Brabant,1628,1628-07-05,437,455


In [ ]:
def unique_texts(series):
    return sorted({str(value) for value in series if pd.notna(value) and str(value).strip() and str(value).strip().lower() != 'nan'})

resolution_window = df_res[['date']].reset_index().rename(columns={'index': 'resolution_id'}).copy()
resolution_window['date'] = pd.PeriodIndex(resolution_window['date'].astype(str), freq='D')
resolution_window = resolution_window[resolution_window['date'].dt.year.between(anchor_window_start, anchor_window_end)].copy()
resolution_window = resolution_window.sort_values(['date', 'resolution_id'])
resolution_window['date_period'] = resolution_window['date']
resolution_window['sequence_nr'] = resolution_window.groupby('date_period').cumcount() + 1
resolution_window['resolution_ref'] = (
    resolution_window['date_period'].astype(str)
    + '-'
    + resolution_window['sequence_nr'].astype(str).str.zfill(2)
)

place_annotations = all_places[all_places.notna()].to_frame().reset_index()
place_annotations.columns = ['resolution_id', 'place_id']
place_annotations['resolution_id'] = place_annotations['resolution_id'].astype(int)
place_annotations['place_id'] = place_annotations['place_id'].astype(str)

place_window = place_annotations.merge(
    df_loc[['id', 'name']],
    how='left',
    left_on='place_id',
    right_on='id',
)
place_window.drop(columns=['id'], inplace=True)
place_window['anchor_label'] = place_window['name'].fillna(place_window['place_id'])
place_window = place_window.merge(
    resolution_window[['resolution_id', 'resolution_ref', 'date_period']],
    on='resolution_id',
    how='inner',
)
place_window['anchor_ref'] = place_window['resolution_ref']

org_annotations = inst_matched.copy()
if 'resolution_id' not in org_annotations.columns:
    org_annotations = org_annotations.reset_index()
    if 'index' in org_annotations.columns and 'resolution_id' not in org_annotations.columns:
        org_annotations = org_annotations.rename(columns={'index': 'resolution_id'})
org_annotations['resolution_id'] = org_annotations['resolution_id'].astype(int)
org_annotations = org_annotations[org_annotations['naam'].notna()].copy()

org_window = org_annotations.merge(
    resolution_window[['resolution_id', 'resolution_ref', 'date_period']],
    on='resolution_id',
    how='inner',
)
org_window['anchor_ref'] = org_window['resolution_ref']
org_window['anchor_label'] = org_window['naam']

place_grouped = place_window.groupby(['resolution_id', 'anchor_ref', 'date_period'], as_index=False).agg(
    place_tags=('anchor_label', unique_texts),
    place_count=('place_id', 'nunique'),
)

org_grouped = org_window.groupby(['resolution_id', 'anchor_ref', 'date_period'], as_index=False).agg(
    org_tags=('anchor_label', unique_texts),
    org_count=('institution_id', 'nunique'),
)

combined_anchor_candidates = place_grouped.merge(
    org_grouped,
    on=['resolution_id', 'anchor_ref', 'date_period'],
    how='inner',
)
combined_anchor_candidates['combined_anchor_score'] = [
    (int(place_count) * 10) + (int(org_count) * 10) + len(place_tags) + len(org_tags)
    for place_count, org_count, place_tags, org_tags in zip(
        combined_anchor_candidates['place_count'],
        combined_anchor_candidates['org_count'],
        combined_anchor_candidates['place_tags'],
        combined_anchor_candidates['org_tags'],
    )
]

print(f"Combined place/org anchor candidates in {anchor_window_start}-{anchor_window_end}: {len(combined_anchor_candidates)}")
print(f"Unique resolution refs: {combined_anchor_candidates['anchor_ref'].nunique()}")
print(f"Unique resolutions: {combined_anchor_candidates['resolution_id'].nunique()}")
combined_anchor_candidates.sort_values(
    ['combined_anchor_score', 'place_count', 'org_count', 'resolution_id'],
    ascending=[False, False, False, True],
)[['resolution_id', 'anchor_ref', 'place_count', 'org_count', 'place_tags', 'org_tags']].head(10)


Combined place/org anchor candidates in 1626-1630: 385
Unique resolution refs: 385
Unique resolutions: 385


,resolution_id,anchor_ref,place_count,org_count,place_tags,org_tags
113,276,1630-04-13-10,12,1,"[Berg, Emmerik, Friesland, Groningen, Kleef, M...",[Generaliteitsrekenkamer]
61,176,1630-04-06-20,9,1,"[Breda, Duisburg, Gravenhage, Hertogenbosch, K...",[Generaliteitsrekenkamer]
52,161,1630-04-06-05,8,1,"[Berg, Büderich, Duisburg, Essen, Lippe, Mark,...",[Generaliteitsrekenkamer]
139,312,1630-04-28-04,7,1,"[Friesland, Gelderland, Groningen, Holland, Ov...",[Generaliteitsrekenkamer]
141,314,1630-04-28-06,7,1,"[Hertogenbosch, Holland, Olinda, Pernambuco, V...",[Generaliteitsrekenkamer]
90,232,1630-04-20-15,6,1,"[Amsterdam, Holland, Oostzee, Portugal, Spanje...",[Generaliteitsrekenkamer]
275,665,1630-03-28-01,6,1,"[Breda, Demer, Hertogenbosch, Heusden, Meierij...",[Raad van State]
306,730,1630-03-16-13,6,1,"[Bergen op Zoom, Halsteren, Oude Land, Sint\n\...",[Raad van State]
262,640,1630-03-26-14,5,1,"[Alicante, Cadiz, Cartagena, Málaga, Sint Lucas]",[Raad van State]
267,648,1630-03-19-02,5,1,"[Delfzijl, Eems, Groningen, Lauwers, Ommelanden]",[Raad van State]


In [119]:
from pathlib import Path
import sys
from tqdm import tqdm

sys.path.append(str(Path(basedir) / "generate_alignment"))
from build_alignment_artifacts import run

# Generate the verify_ground_truth outputs from the notebook environment.
tqdm.pandas(desc="Generating verify_ground_truth outputs")
run(preview_limit=10, stratified_size=100, verification_page_size=30)

print("Generated verify_ground_truth outputs in:", output_dir)

Loading data files...
✓ Loaded 19134 enriched resolutions
✓ Loaded 692156 flat resolutions
✓ Loaded 2459 LOC canonical names
✓ Loaded 8076 PER canonical names
✓ Loaded 341 ORG canonical names
✓ Anchor map: 1579 confirmed date anchors
✓ Found 10 preview matches with place/org overlap
  Place/org enriched candidates scanned: 2000
  Place-only fallback candidates scanned: 2000
  Summary anchors: 6
✓ HTML report saved to: /Users/rikhoekstra/develop/republic_ner_matching/output/matched_resolutions_sample.html

📅 Stratified sampling across corpus:
   Months covered: 53
   Items per month: 1
  Progress: 50/53
✓ Stratified matches: 45
  Summary anchors: 30
✅ Ground truth exported: 45 samples
  Summary anchors: 30
  Entity counts: {'places': {'enriched': 554, 'found_in_flat': 146}, 'persons': {'enriched': 233, 'found_in_flat': 0}, 'organizations': {'enriched': 138, 'found_in_flat': 14}}
 sample_id enriched_date  flat_date  date_diff_days  is_same_day  is_summary_anchor  confidence_score  place_